In [28]:
import os
from dotenv import load_dotenv
import hopsworks
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
# Import your specific model here (e.g., from xgboost import XGBRegressor)

# Load your local environment variables
load_dotenv()

print("Connecting to Hopsworks...")
project = hopsworks.login(
    host="eu-west.cloud.hopsworks.ai", 
    project="aqi_prediction_project", 
    api_key_value=os.getenv("HOPSWORKS_API_KEY"),
    hostname_verification=False       
)
fs = project.get_feature_store()

# Fetching the exact Feature View your global model uses!
print("Fetching Feature View...")
feature_view = fs.get_feature_view(name="global_aqi_view", version=5)
print("Connected successfully!")

Connecting to Hopsworks...
2026-05-08 18:04:26,702 INFO: Closing external client and cleaning up certificates.
2026-05-08 18:04:26,762 INFO: Connection closed.
2026-05-08 18:04:26,774 INFO: Initializing external client
2026-05-08 18:04:26,776 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-05-08 18:04:30,628 INFO: Python Engine initialized.

Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/31963
Fetching Feature View...
Connected successfully!


In [29]:
print("Downloading historical dataset from Feature View...")
# Read through the ML Lens
df = feature_view.get_batch_data()

print(f"Original shape (8 cities): {df.shape}")

# EXPERIMENT: Isolate Karachi
karachi_df = df[df['city'].str.lower() == 'karachi'].copy()

# Drop the city column (it's a constant now, which ML models hate)
karachi_df = karachi_df.drop(columns=['city'])

# Sort by date to ensure chronological integrity
karachi_df = karachi_df.sort_values('date').reset_index(drop=True)

print(f"Filtered shape (Karachi only): {karachi_df.shape}")

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (75.64s) 
Original shape (8 cities): (8824, 18)
Filtered shape (Karachi only): (1103, 17)


In [30]:
# Assuming we are predicting tomorrow's PM2.5 (target_pm2_5_1d)
# You can change this if you want to predict 2 days or 3 days out
target_col = 'target_pm2_5_1d'

# Drop all target columns from our input features
feature_cols = [col for col in karachi_df.columns if not col.startswith('target_') and col != 'date']

# Drop any rows where our specific target is NaN
karachi_df = karachi_df.dropna(subset=[target_col])

X = karachi_df[feature_cols]
y = karachi_df[target_col]

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

Features (X) shape: (1103, 13)
Target (y) shape: (1103,)


In [31]:
# Let's use the first 80% of time for training, and the most recent 20% for testing
split_index = int(len(karachi_df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Keep the dates for plotting later if needed
dates_test = karachi_df['date'].iloc[split_index:]

print(f"Training on {len(X_train)} days, testing on {len(X_test)} days.")

Training on 882 days, testing on 221 days.


In [32]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

print("Starting Model Zoo Training Loop...\n")

# Use TimeSeriesSplit to prevent time-leakage during cross-validation
tscv = TimeSeriesSplit(n_splits=3)

# Dictionaries to store our trained models and their scores
trained_models = {}
results = []

# Loop through every model in the zoo
for model_name, config in model_zoo.items():
    print(f"--- Training {model_name} ---")
    
    # Set up the Grid Search
    grid_search = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=tscv,  # Use chronological splitting
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )
    
    # Train the grid search to find the best parameters
    grid_search.fit(X_train, y_train)
    
    # Extract the absolute best version of this model
    best_model = grid_search.best_estimator_
    trained_models[model_name] = best_model
    
    # Predict on our unseen 20% test set
    predictions = best_model.predict(X_test)
    
    # Calculate performance metrics
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    
    print(f"Best Params: {grid_search.best_params_}")
    print(f"Test MAE:  {mae:.2f}")
    print(f"Test RMSE: {rmse:.2f}\n")
    
    # Save the results for the leaderboard
    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse
    })

# Print a nice leaderboard to see who won
print("=== FINAL LEADERBOARD ===")
leaderboard = pd.DataFrame(results).sort_values(by="MAE").reset_index(drop=True)
print(leaderboard)

Starting Model Zoo Training Loop...

--- Training Ridge_Regression ---
Best Params: {'alpha': 100.0}
Test MAE:  5.91
Test RMSE: 8.04

--- Training XGBoost ---
Best Params: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300}
Test MAE:  5.99
Test RMSE: 8.00

--- Training LightGBM ---
Best Params: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 31}
Test MAE:  6.18
Test RMSE: 8.34

=== FINAL LEADERBOARD ===
              Model       MAE      RMSE
0  Ridge_Regression  5.908369  8.042021
1           XGBoost  5.989025  8.000372
2          LightGBM  6.180140  8.337201


In [33]:
# 1. Find the name of the best model from our leaderboard
best_model_name = leaderboard.iloc[0]['Model']
print(f"🥇 The winner is: {best_model_name}!")

# 2. Extract that specific trained model from our dictionary
winning_model = trained_models[best_model_name]

# 3. Predict on the unseen 20% using the winner
predictions = winning_model.predict(X_test)

# 4. Calculate final metrics
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("\n--- KARACHI LOCALIZED MODEL (WINNER) PERFORMANCE ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

# TIP: Write these numbers down! You will compare them to your Global Model's performance next.

🥇 The winner is: Ridge_Regression!

--- KARACHI LOCALIZED MODEL (WINNER) PERFORMANCE ---
Mean Absolute Error (MAE): 5.91
Root Mean Squared Error (RMSE): 8.04


In [35]:
import joblib
import os

print("Fetching the Global Model from Hopsworks...")
mr = project.get_model_registry()

# Fetching your specific Champion model
global_model_metadata = mr.get_model("aqi_target_pm2_5_1d_model", version=27) 
model_dir = global_model_metadata.download()

# Load the saved global model
global_model = joblib.load(model_dir + "/target_pm2_5_1d_model.pkl") 
print("Global model loaded successfully!\n")

# --- RECONSTRUCT X_TEST FOR THE GLOBAL MODEL ---
# Based on your logs, the global model expects these exact columns in this exact order:
global_features = [
    'pm10', 'pm2_5', 'no2', 'ozone', 'temperature_2m', 'precipitation', 
    'wind_speed_10m', 'month', 'day_of_week', 'day_of_year', 
    'pm2_5_rolling_3d', 'pm2_5_rolling_7d', 'pm2_5_change_rate'
]

# Slice our Karachi test set to match the exact feature order the global model wants
X_test_global = X_test[global_features]

# Predict using the Global Model
print("Evaluating Global Model specifically on Karachi's future data...")
global_predictions = global_model.predict(X_test_global)

global_mae = mean_absolute_error(y_test, global_predictions)
global_rmse = np.sqrt(mean_squared_error(y_test, global_predictions))

print("\n Comparison")
print("----------------------------------------")
print(f"KARACHI-ONLY MODEL (Ridge):")
print(f"  MAE:  {mae:.2f}")
print(f"  RMSE: {rmse:.2f}")
print("----------------------------------------")
print(f"GLOBAL 8-CITY MODEL:")
print(f"  MAE:  {global_mae:.2f}")
print(f"  RMSE: {global_rmse:.2f}")
print("----------------------------------------")

# The Verdict
if mae < global_mae:
    print("\nThe Karachi only model performs better")
else:
    print("\n The 8 City model performs better")

Fetching the Global Model from Hopsworks...


Downloading: 0.000%|          | 0/274891 elapsed<00:00 remaining<?

Global model loaded successfully!s, 1 files)... DONE

Evaluating Global Model specifically on Karachi's future data...

 Comparison
----------------------------------------
KARACHI-ONLY MODEL (Ridge):
  MAE:  5.91
  RMSE: 8.04
----------------------------------------
GLOBAL 8-CITY MODEL:
  MAE:  6.11
  RMSE: 8.18
----------------------------------------

The Karachi only model performs better
